# 5. Agentic Pattern - Tool Use (Function Calling)
The Tool Use pattern, often implemented through a mechanism called Function Calling, enables an agent to interact with external APIs, databases, services, or even execute code.

[![Agentic Pattern - Tool Use presentation](https://img.youtube.com/vi/?/0.jpg)](https://youtu.be/?) 

<br />
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<a href="https://www.youtube.com/@dayonedev" target="_new">
  <img align="center" src="https://img.shields.io/youtube/channel/views/UCiLziPE9aPxCouSsX0lJ--A" />
</a>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<a href="https://linkedin.com/in/krishnamanchikalapudi" target="_new">
  <img align="center" src="https://img.shields.io/badge/linkedin-%230077B5.svg?style=for-the-badge&logo=linkedin&logoColor=white" />
</a>
<br/>

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 6.54 ms, sys: 8.21 ms, total: 14.7 ms
Wall time: 922 ms


In [2]:
%pip install -U -q langchain langchain-core langchain-ollama ipython-autotime --use-deprecated=legacy-resolver

%load_ext autotime

Note: you may need to restart the kernel to use updated packages.
time: 162 μs (started: 2025-12-04 12:57:41 -08:00)


## Variables

In [3]:
my_model_ollama = "llama3.2"

time: 212 μs (started: 2025-12-04 12:57:41 -08:00)


## Initialize OLLAM service
OLLAMA inference client is initialized to interact with the OLLAMA API for generating responses from the specified model.

In [4]:
from langchain_ollama.llms import OllamaLLM

llm_client = OllamaLLM(
    model=my_model_ollama,
    base_url="http://localhost:11434",
    headers={"Content-Type": "application/json"},
    stream=True,
    temperature=0.7,
)


llm_client

OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')

time: 1.81 s (started: 2025-12-04 12:57:41 -08:00)


### TEST llm_client with a simple prompt

In [5]:
response = llm_client.invoke("What is Agentic Pattern - Tool Use")

print(f"{response}")

Agentic Pattern - Tool Use (AP-TU) is a theory of cognitive development and evolution that was proposed by Dr. Alison Gopnik, Dr. Andrew N. Meltzoff, and Dr. Patricia K. Kuhl in 2007.

According to AP-TU, children's ability to use tools is not just a matter of cognitive development, but also of social interaction and learning through observation. The theory posits that children learn to use tools by observing others, including adults and older children, and then practicing and imitating their behavior.

The key idea behind AP-TU is that children's first experiences with tool use are often centered around play, where they watch and imitate others using objects in creative ways. As they observe and practice, they begin to develop a sense of agency and control over the tools and objects around them.

AP-TU suggests that tool use is an essential aspect of human development, as it provides children with opportunities for exploration, creativity, and problem-solving. The theory also highligh

## Define a Tool functions 

In [6]:
from langchain_core.tools import tool as langchain_tool


@langchain_tool
def calculator(expression: str) -> str:
    """Evaluate a simple math expression and return the result."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

time: 7.57 ms (started: 2025-12-04 12:57:49 -08:00)


In [7]:
@langchain_tool
def get_weather(location: str) -> str:
    """Get the current weather for a location (stub implementation)."""
    fake_weather = {
        "London": "Rainy, 18°C",
        "New York": "Sunny, 25°C",
        "Tokyo": "Cloudy, 22°C",
        "San Francisco": "Cloudy, 20°C",
    }
    return fake_weather.get(location, "Weather data not available")

time: 4.07 ms (started: 2025-12-04 12:57:49 -08:00)


In [8]:
@langchain_tool
def search_information(query: str) -> str:
    """
    Provides factual information on a given topic. Use this tool to find answers to phrases
    like 'capital of France' or 'weather in London?'.
    """
    print(f"\n--- 🛠️ Tool Called: search_information with query: '{query}' ---")
    simulated_results = {
        "weather in london": "The weather in London is currently cloudy with a temperature of 15°C.",
        "capital of france": "The capital of France is Paris.",
        "population of earth": "The estimated population of Earth is around 8 billion people.",
        "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
        "default": f"Simulated search result for '{query}': No specific information found, but the topic seems interesting.",
    }
    result = simulated_results.get(query.lower(), simulated_results["default"])
    print(f"--- TOOL RESULT: {result} ---")
    return result

time: 2.43 ms (started: 2025-12-04 12:57:49 -08:00)


## Bind the Tool to the Model

In [9]:
# llm_with_tools = [add_two_numbers, multiply, get_weather, search_information]

llm_with_tools = {"calculator": calculator, "get_weather": get_weather}

llm_with_tools

{'calculator': StructuredTool(name='calculator', description='Evaluate a simple math expression and return the result.', args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x3184072e0>),
 'get_weather': StructuredTool(name='get_weather', description='Get the current weather for a location (stub implementation).', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x318461080>)}

time: 1.72 ms (started: 2025-12-04 12:57:49 -08:00)


In [10]:
from langchain_core.prompts import ChatPromptTemplate

agent_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI assistant. You can use tools when needed."),
        ("human", "{input}"),
    ]
)
agent_prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful AI assistant. You can use tools when needed.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

time: 30 ms (started: 2025-12-04 12:57:49 -08:00)


### Router Logic: detect if tool call is required

In [11]:
import json


def route_tool_call(response: str):
    """
    Very simple parser: expects the model to output JSON like:
    {"tool": "calculator", "args": {"expression": "25*4+10"}}
    If no tool call, return as final answer.
    """
    try:
        data = json.loads(response)
        if "tool" in data and data["tool"] in llm_with_tools:
            tool_func = llm_with_tools[data["tool"]]
            return tool_func.run(**data["args"])
    except Exception:
        return response
    return response

time: 589 μs (started: 2025-12-04 12:57:49 -08:00)


In [12]:
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough

chain = (
    {"input": RunnablePassthrough()}
    | agent_prompt
    | llm_client
    | RunnableLambda(route_tool_call)
)

chain

ModuleNotFoundError: No module named 'langchain.schema'

time: 87.2 ms (started: 2025-12-04 12:57:49 -08:00)


## Invoke the Model with a Prompt

In [ ]:
response = chain.invoke("What is the weather in London?")

print(f"{response}")

I'd be happy to help you with that! However, I'm a large language model, I don't have real-time access to current weather conditions. But I can suggest some ways for you to find out the latest weather forecast for London.

You can check online weather websites such as:

* BBC Weather
* Met Office
* AccuWeather

These websites provide up-to-date weather forecasts, including temperature, humidity, wind speed, and precipitation. You can also download mobile apps like Dark Sky or Weather Underground that provide hyperlocal weather forecasts.

If you're looking for a more personalized experience, I can suggest some popular weather-related tools and resources:

* Google Search: Type "London weather" in Google to get the latest forecast.
* Weather Widget: Install a weather widget on your phone or computer to display the current weather conditions.
* Smart Speaker: Ask your smart speaker like Alexa or Google Assistant about the London weather.

Please let me know if there's anything else I can

In [ ]:
response = chain.invoke(
    "Please calculate 25 * 4 + 10 using the calculator tool. Output JSON with tool call."
)

print(f"{response}")

Here is the calculation in JSON format:

```
{
  "tool": "calculator",
  "calculation": "25 * 4 + 10"
}
```

And here's the result of the calculation:

```
25 * 4 = 100
100 + 10 = 110
```
time: 1.38 s (started: 2025-09-24 17:38:33 -07:00)


In [ ]:
response = chain.invoke(
    "If I bought 100 shares of Apple at $190, and now price is $275, use calculator to compute my profit."
)

print(f"{response}")

To calculate your profit, we'll need to find the difference between the original price and the current price.

Original price = $190
Current price = $275

Profit = Current price - Original price
= $275 - $190
= $85

So, if you bought 100 shares of Apple at $190 and now the price is $275, your profit would be $85 per share.
time: 1.65 s (started: 2025-09-24 17:38:34 -07:00)
